In [30]:
import pandas as pd
import numpy as np
import os
import time
from abc import ABC, abstractmethod
from functools import wraps

## Exercise 1: Build the `CSVReader` Class (OOP Foundation)

In [31]:
class CSVReader:
    def __init__(self, file_path):
        self.file_path = file_path
        self._data = None

    def read(self):
        if not os.path.exists(self.file_path):
            raise FileNotFoundError(f"File not found: {self.file_path}")
        self._data = pd.read_csv(self.file_path)
        return self._data

    def preview(self, n=5):
        if self._data is None:
            self.read()
        print(f"Top {n} rows of '{self.file_path}':")
        print(self._data.head(n))

    @property
    def data(self):
        return self._data

    def __repr__(self):
        status = "loaded" if self._data is not None else "not loaded"
        return f"CSVReader(file_path='{self.file_path}', status={status})"

### Test the class with `sample_data.csv`

In [32]:
reader = CSVReader("sample_data.csv")
print(reader)

CSVReader(file_path='sample_data.csv', status=not loaded)


In [33]:
df = reader.read()
print(reader)
print("Shape:", df.shape)
df

CSVReader(file_path='sample_data.csv', status=loaded)
Shape: (7, 7)


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,NaN
6,7,Grace,NaN,162.0,60.0,Chicago,92.0


In [34]:
reader.preview(3)

Top 3 rows of 'sample_data.csv':
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN


In [35]:
print(type(reader.data))
print("Missing values per column:")
print(reader.data.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
Missing values per column:
id           0
name         0
age          3
height_cm    1
weight_kg    1
city         0
score        2
dtype: int64


In [36]:
try:
    CSVReader("not_exist.csv").read()
except FileNotFoundError as e:
    print("Error:", e)

Error: File not found: not_exist.csv


## Exercise 2: Apply Pattern for Missing Values Handling

In [37]:
class MissingValueStrategy(ABC):
    @abstractmethod
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        pass

In [38]:
class DropMissing(MissingValueStrategy):
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.dropna().reset_index(drop=True)

In [39]:
class FillMean(MissingValueStrategy):
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        result = df.copy()
        numeric_cols = result.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            result[col] = result[col].fillna(result[col].mean())
        return result

In [40]:
class FillMode(MissingValueStrategy):
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        result = df.copy()
        for col in result.columns:
            if result[col].isnull().any():
                mode_val = result[col].mode()
                if not mode_val.empty:
                    result[col] = result[col].fillna(mode_val[0])
        return result

In [41]:
class DataCleaner:
    def __init__(self, strategy: MissingValueStrategy):
        self.strategy = strategy

    def set_strategy(self, strategy: MissingValueStrategy):
        self.strategy = strategy

    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        return self.strategy.handle(df)

### Test Missing Value Strategies

In [42]:
cleaner = DataCleaner(DropMissing())
df_dropped = cleaner.clean(df)
df_dropped

,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,5,Eva,27.0,160.0,55.0,New York,88.0


In [43]:
cleaner.set_strategy(FillMean())
df_mean = cleaner.clean(df)
df_mean

,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.00,165.0,68.0,New York,85.0
1,2,Bob,30.25,172.0,70.0,Los Angeles,90.0
2,3,Charlie,35.00,168.0,72.0,Chicago,86.0
3,4,David,30.25,167.0,80.0,Houston,75.0
4,5,Eva,27.00,160.0,55.0,New York,88.0
5,6,Frank,30.00,175.0,85.0,Los Angeles,86.0
6,7,Grace,30.25,162.0,60.0,Chicago,92.0


In [44]:
cleaner.set_strategy(FillMode())
df_mode = cleaner.clean(df)
df_mode

,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,27.0,172.0,55.0,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,75.0
3,4,David,27.0,160.0,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,75.0
6,7,Grace,27.0,162.0,60.0,Chicago,92.0


## Exercise 3: Add Decorators for Logging and Timing

In [45]:
def log_action(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[LOG] Starting '{func.__name__}'...")
        result = func(*args, **kwargs)
        print(f"[LOG] Finished '{func.__name__}'.")
        return result
    return wrapper

In [46]:
def log_time(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start_time
        print(f"[TIME] '{func.__name__}' executed in {elapsed:.6f}s")
        return result
    return wrapper

In [47]:
class CSVReader:
    def __init__(self, file_path):
        self.file_path = file_path
        self._data = None

    @log_action
    @log_time
    def read(self):
        if not os.path.exists(self.file_path):
            raise FileNotFoundError(f"File not found: {self.file_path}")
        self._data = pd.read_csv(self.file_path)
        return self._data

    @log_action
    def preview(self, n=5):
        if self._data is None:
            self.read()
        print(f"Top {n} rows of '{self.file_path}':")
        print(self._data.head(n))

In [48]:
class DataCleaner:
    def __init__(self, strategy: MissingValueStrategy):
        self.strategy = strategy

    def set_strategy(self, strategy: MissingValueStrategy):
        self.strategy = strategy

    @log_action
    @log_time
    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        return self.strategy.handle(df)

### Test Decorators on `CSVReader` and `DataCleaner`

In [49]:
reader = CSVReader("sample_data.csv")
df = reader.read()
reader.preview(3)

cleaner = DataCleaner(FillMean())
cleaned_df = cleaner.clean(df)
cleaned_df

[LOG] Starting 'read'...
[TIME] 'read' executed in 0.001302s
[LOG] Finished 'read'.
[LOG] Starting 'preview'...
Top 3 rows of 'sample_data.csv':
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
[LOG] Finished 'preview'.
[LOG] Starting 'clean'...
[TIME] 'clean' executed in 0.000863s
[LOG] Finished 'clean'.


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.00,165.0,68.0,New York,85.0
1,2,Bob,30.25,172.0,70.0,Los Angeles,90.0
2,3,Charlie,35.00,168.0,72.0,Chicago,86.0
3,4,David,30.25,167.0,80.0,Houston,75.0
4,5,Eva,27.00,160.0,55.0,New York,88.0
5,6,Frank,30.00,175.0,85.0,Los Angeles,86.0
6,7,Grace,30.25,162.0,60.0,Chicago,92.0


## Exercise 4: Implement Factory Pattern for Data Transformations

In [50]:
class DataTransform(ABC):
    @abstractmethod
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        pass

In [51]:
class NormalizeColumns(DataTransform):
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df.columns = [col.strip().lower().replace(" ", "_") for col in df.columns]
        return df

In [52]:
class RemoveDuplicates(DataTransform):
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.drop_duplicates().reset_index(drop=True)

In [53]:
class StandardizeText(DataTransform):
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        text_cols = df.select_dtypes(include=["object", "string"]).columns
        for col in text_cols:
            df[col] = df[col].str.strip().str.lower()
        return df

In [54]:
class TransformFactory:
    @staticmethod
    def create_transform(transform_type: str) -> DataTransform:
        transforms = {
            "normalize_columns": NormalizeColumns,
            "remove_duplicates": RemoveDuplicates,
            "standardize_text": StandardizeText,
        }
        key = transform_type.strip().lower()
        if key not in transforms:
            raise ValueError(f"Unknown transformation: '{transform_type}'")
        return transforms[key]()

### Test `TransformFactory`

In [55]:
df_transformed = cleaned_df.copy()
steps = ["normalize_columns", "standardize_text", "remove_duplicates"]

for step in steps:
    transformer = TransformFactory.create_transform(step)
    df_transformed = transformer.apply(df_transformed)

df_transformed

,id,name,age,height_cm,weight_kg,city,score
0,1,alice,29.00,165.0,68.0,new york,85.0
1,2,bob,30.25,172.0,70.0,los angeles,90.0
2,3,charlie,35.00,168.0,72.0,chicago,86.0
3,4,david,30.25,167.0,80.0,houston,75.0
4,5,eva,27.00,160.0,55.0,new york,88.0
5,6,frank,30.00,175.0,85.0,los angeles,86.0
6,7,grace,30.25,162.0,60.0,chicago,92.0


## Exercise 5: Build a Full Cleaning Pipeline (Template Method Pattern)

In [56]:
class DataPipeline(ABC):
    @log_action
    @log_time
    def run(self):
        self.load()
        self.clean()
        self.transform()
        self.save()
        return self.df

    @abstractmethod
    def load(self):
        pass

    @abstractmethod
    def clean(self):
        pass

    @abstractmethod
    def transform(self):
        pass

    @abstractmethod
    def save(self):
        pass

In [57]:
class CSVDataPipeline(DataPipeline):
    def __init__(self, input_path: str, output_path: str, missing_strategy: MissingValueStrategy, transforms: list[str]):
        self.input_path = input_path
        self.output_path = output_path
        self.reader = CSVReader(input_path)
        self.cleaner = DataCleaner(missing_strategy)
        self.transforms = transforms
        self.df = None

    @log_action
    def load(self):
        self.df = self.reader.read()

    @log_action
    def clean(self):
        self.df = self.cleaner.clean(self.df)

    @log_action
    @log_time
    def transform(self):
        for step in self.transforms:
            transformer = TransformFactory.create_transform(step)
            self.df = transformer.apply(self.df)

    @log_action
    def save(self):
        self.df.to_csv(self.output_path, index=False)
        print(f"Saved cleaned data to '{self.output_path}'")

### Test `CSVDataPipeline`

In [58]:
pipeline = CSVDataPipeline(
    input_path="sample_data.csv",
    output_path="cleaned_sample_data.csv",
    missing_strategy=FillMean(),
    transforms=["normalize_columns", "standardize_text", "remove_duplicates"]
)

final_df = pipeline.run()
final_df

[LOG] Starting 'run'...
[LOG] Starting 'load'...
[LOG] Starting 'read'...
[TIME] 'read' executed in 0.001461s
[LOG] Finished 'read'.
[LOG] Finished 'load'.
[LOG] Starting 'clean'...
[LOG] Starting 'clean'...
[TIME] 'clean' executed in 0.000792s
[LOG] Finished 'clean'.
[LOG] Finished 'clean'.
[LOG] Starting 'transform'...
[TIME] 'transform' executed in 0.001536s
[LOG] Finished 'transform'.
[LOG] Starting 'save'...
Saved cleaned data to 'cleaned_sample_data.csv'
[LOG] Finished 'save'.
[TIME] 'run' executed in 0.004821s
[LOG] Finished 'run'.


,id,name,age,height_cm,weight_kg,city,score
0,1,alice,29.00,165.0,68.0,new york,85.0
1,2,bob,30.25,172.0,70.0,los angeles,90.0
2,3,charlie,35.00,168.0,72.0,chicago,86.0
3,4,david,30.25,167.0,80.0,houston,75.0
4,5,eva,27.00,160.0,55.0,new york,88.0
5,6,frank,30.00,175.0,85.0,los angeles,86.0
6,7,grace,30.25,162.0,60.0,chicago,92.0
